# Run x04: the standard Scanpy workflow on 2,700 blood cells

An exploration from the Virtual Cell Lab. It asks whether a computer can find cell types nobody labelled, and how you tell a real
difference in a gene from luck. The predictions are on the run card, committed before this notebook was executed. No model is scored.

Data: *3k PBMCs from a Healthy Donor*, 10x Genomics. Tools: Scanpy (Wolf, Angerer & Theis, Genome Biology 2018).

To run it yourself in a browser tab, open it in Google Colab and choose Run all. It downloads about 7.6 MB.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip install -q scanpy igraph leidenalg

## 1. Get the file, and check it is the same file we used

In [2]:
import hashlib, json, os, subprocess, tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.metrics import adjusted_rand_score

SEED = 13
URL = "https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"
SHA256 = "847d6ebd9a1ec9a768f2be7e40ca42cbfe75ebeb6d76a4c24167041699dc28b5"
DATA = Path(os.environ.get("X04_DATA", "data"))
OUT = Path(os.environ.get("X04_OUT", "."))
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

archive = DATA / "pbmc3k_filtered_gene_bc_matrices.tar.gz"
if not archive.exists():
    subprocess.run(["curl", "-fsSL", "-o", str(archive), URL], check=True)  # the server refuses Python's default downloader
assert hashlib.sha256(archive.read_bytes()).hexdigest() == SHA256, "not the file this run used"
with tarfile.open(archive) as t:
    t.extractall(DATA, filter="data")

adata = sc.read_10x_mtx(DATA / "filtered_gene_bc_matrices" / "hg19", var_names="gene_symbols", cache=False)
adata.var_names_make_unique()
adata

AnnData object with n_obs × n_vars = 2700 × 32738
    var: 'gene_ids'
    layers: None (.X)

## 2. How empty is it?

In [3]:
results = {"run": "x04-scanpy-walkthrough", "versions": {"scanpy": sc.__version__}}
cells, genes = adata.shape
entries = cells * genes
zero_share = 1 - adata.X.nnz / entries
results["raw"] = {"cells": cells, "genes": genes, "entries": entries, "zero_pct": round(100 * zero_share, 2)}
results["raw"]

{'cells': 2700, 'genes': 32738, 'entries': 88392600, 'zero_pct': 97.41}

## 3. Throw out the bad cells

In [4]:
sc.pp.filter_genes(adata, min_cells=3)
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
before = adata.n_obs
keep = (adata.obs.n_genes_by_counts > 200) & (adata.obs.n_genes_by_counts < 2500) & (adata.obs.pct_counts_mt < 5)
adata = adata[keep, :].copy()
results["qc"] = {"cells_before": before, "cells_after": adata.n_obs, "genes_after": adata.n_vars,
                 "median_genes_per_cell": int(adata.obs.n_genes_by_counts.median())}
adata.layers["counts"] = adata.X.copy()
results["qc"]

{'cells_before': 2700,
 'cells_after': 2638,
 'genes_after': 13714,
 'median_genes_per_cell': 819}

## 4. Normalize, pick the genes that vary, and squash 2,000 dimensions down

In [5]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
scaled = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(scaled, max_value=10)
sc.pp.pca(scaled, n_comps=50, svd_solver="arpack", random_state=SEED)
sc.pp.neighbors(scaled, n_neighbors=10, n_pcs=40, random_state=SEED)
sc.tl.umap(scaled, random_state=SEED)
variance = scaled.uns["pca"]["variance_ratio"]
results["pca"] = {"hvg": int(adata.var.highly_variable.sum()),
                  "variance_pct_first_2": round(100 * float(variance[:2].sum()), 2),
                  "variance_pct_first_40": round(100 * float(variance[:40].sum()), 2)}
results["pca"]

{'hvg': 2000, 'variance_pct_first_2': 3.79, 'variance_pct_first_40': 12.45}

## 5. Let the cells sort themselves

In [6]:
def leiden(resolution, seed):
    key = f"leiden_{resolution}_{seed}"
    sc.tl.leiden(scaled, resolution=resolution, random_state=seed, flavor="igraph", n_iterations=2, directed=False, key_added=key)
    return scaled.obs[key].astype(str).to_numpy()

labels = leiden(0.7, SEED)
adata.obs["cluster"] = pd.Categorical(labels)
sizes = adata.obs["cluster"].value_counts().sort_values(ascending=False)
results["clusters"] = {"resolution": 0.7, "n": int(sizes.size), "sizes": {str(k): int(v) for k, v in sizes.items()}}
results["clusters"]

{'resolution': 0.7,
 'n': 7,
 'sizes': {'0': 1167,
  '1': 442,
  '4': 417,
  '2': 341,
  '3': 219,
  '5': 37,
  '6': 15}}

## 6. Do the clusters mean anything? Five genes earlier work ties to blood cell types

In [7]:
MARKERS = {"MS4A1": "B cells", "CD3D": "T cells", "LYZ": "monocytes", "NKG7": "NK and cytotoxic T cells", "PPBP": "platelets"}
detected = adata.layers["counts"] > 0
rows = []
for gene, cell_type in MARKERS.items():
    hit = np.asarray(detected[:, adata.var_names.get_loc(gene)].todense()).ravel()
    per_cluster = pd.Series(hit).groupby(labels).mean()
    best = per_cluster.idxmax()
    in_best, elsewhere = hit[labels == best].mean(), hit[labels != best].mean()
    rows.append({"gene": gene, "expected": cell_type, "best_cluster": best, "pct_in_best": round(100 * float(in_best), 1),
                 "pct_elsewhere": round(100 * float(elsewhere), 1),
                 "fold": round(float(in_best / elsewhere), 2) if elsewhere else None})
results["markers"] = rows
pd.DataFrame(rows)

,gene,expected,best_cluster,pct_in_best,pct_elsewhere,fold
0,MS4A1,B cells,2,85.3,5.4,15.81
1,CD3D,T cells,0,87.4,23.9,3.65
2,LYZ,monocytes,4,100.0,53.0,1.89
3,NKG7,NK and cytotoxic T cells,1,97.3,16.3,5.97
4,PPBP,platelets,6,100.0,2.3,43.00


## 7. Turn the knob: how many cell types are there?

In [8]:
counts_by_resolution = {r: int(pd.unique(leiden(r, SEED)).size) for r in [0.25, 0.5, 0.7, 1.0, 1.5, 2.0]}
results["resolution"] = {str(k): v for k, v in counts_by_resolution.items()}
aris = {s: round(float(adjusted_rand_score(labels, leiden(0.7, s))), 4) for s in range(100, 110)}
results["seeds"] = {"against_seed": SEED, "ari": {str(k): v for k, v in aris.items()}, "min": min(aris.values())}
results["resolution"], results["seeds"]["min"]

({'0.25': 6, '0.5': 6, '0.7': 7, '1.0': 9, '1.5': 14, '2.0': 20}, 0.988)

## 8. A real difference, or luck?

In [9]:
def test(groups, tie_correct=True):
    x = adata.copy()
    x.X = x.layers["lognorm"]
    x.obs["group"] = pd.Categorical(np.where(groups, "a", "b"))
    sc.tl.rank_genes_groups(x, "group", groups=["a"], reference="rest", method="wilcoxon", tie_correct=tie_correct,
                            corr_method="benjamini-hochberg", n_genes=x.n_vars, use_raw=False)
    df = sc.get.rank_genes_groups_df(x, "a")
    return {"genes_tested": int(len(df)), "n_unadjusted_lt_05": int((df.pvals < 0.05).sum()),
            "pct_unadjusted_lt_05": round(100 * float((df.pvals < 0.05).mean()), 2), "n_bh_lt_05": int((df.pvals_adj < 0.05).sum())}, df

ms4a1_best = next(r["best_cluster"] for r in results["markers"] if r["gene"] == "MS4A1")
real_groups = labels == ms4a1_best
shuffled_groups = np.random.default_rng(SEED).permutation(real_groups)

real, real_df = test(real_groups)
shuffled, _ = test(shuffled_groups)
shuffled_default, _ = test(shuffled_groups, tie_correct=False)
results["de"] = {"cluster": ms4a1_best, "cells_in_cluster": int(real_groups.sum()), "real": real, "shuffled": shuffled,
                 "shuffled_no_tie_correction_secondary": shuffled_default,
                 "top_real_genes": real_df.head(10)[["names", "logfoldchanges", "pvals_adj"]].round(4).astype({"names": str}).to_dict("records")}
{k: results["de"][k] for k in ("real", "shuffled", "shuffled_no_tie_correction_secondary")}

{'real': {'genes_tested': 13714,
  'n_unadjusted_lt_05': 3252,
  'pct_unadjusted_lt_05': 23.71,
  'n_bh_lt_05': 2060},
 'shuffled': {'genes_tested': 13714,
  'n_unadjusted_lt_05': 684,
  'pct_unadjusted_lt_05': 4.99,
  'n_bh_lt_05': 7},
 'shuffled_no_tie_correction_secondary': {'genes_tested': 13714,
  'n_unadjusted_lt_05': 30,
  'pct_unadjusted_lt_05': 0.22,
  'n_bh_lt_05': 0}}

## 9. Score our own predictions, and save everything

In [10]:
m = results["markers"]
r = results["resolution"]
s = results["de"]["shuffled"]
held = [
    ("more than 90% zero", results["raw"]["zero_pct"] > 90),
    ("6 to 10 clusters at 0.7", 6 <= results["clusters"]["n"] <= 10),
    ("at least 4 of 5 markers two-fold", sum(1 for x in m if x["fold"] is None or x["fold"] >= 2) >= 4),
    ("resolution 2.0 at least twice 0.25", r["2.0"] >= 2 * r["0.25"]),
    ("all seeds ARI at least 0.8", results["seeds"]["min"] >= 0.8),
    ("shuffled 3-7% and at most 5 pass BH", 3 <= s["pct_unadjusted_lt_05"] <= 7 and s["n_bh_lt_05"] <= 5),
    ("real labels at least 100 pass BH", results["de"]["real"]["n_bh_lt_05"] >= 100),
]
results["predictions"] = [{"statement": a, "held": bool(b)} for a, b in held]

umap = scaled.obsm["X_umap"]
points = [{"x": round(float(a), 3), "y": round(float(b), 3), "c": str(c)} for (a, b), c in zip(umap, labels)]
(OUT / "results.json").write_text(json.dumps(results, indent=2) + "\n")
(OUT / "umap.json").write_text(json.dumps(points) + "\n")

lines = ["# Run x04: the standard Scanpy workflow on 2,700 blood cells", "", "Generated by `walkthrough.ipynb`.", "",
         f"- Raw matrix: {results['raw']['cells']:,} cells by {results['raw']['genes']:,} genes, {results['raw']['zero_pct']}% zero",
         f"- After QC: {results['qc']['cells_after']:,} cells, {results['qc']['genes_after']:,} genes",
         f"- Clusters at resolution 0.7: {results['clusters']['n']}",
         f"- Clusters by resolution: {results['resolution']}",
         f"- Lowest adjusted Rand index across ten seeds: {results['seeds']['min']}", "",
         "| Prediction | Held |", "|---|---|"] + [f"| {p['statement']} | {'yes' if p['held'] else 'NO'} |" for p in results["predictions"]] + [
         "", "| Marker | Expected | Best cluster | % in best | % elsewhere | Fold |", "|---|---|---|---|---|---|"] + [
         f"| {x['gene']} | {x['expected']} | {x['best_cluster']} | {x['pct_in_best']} | {x['pct_elsewhere']} | {x['fold']} |" for x in m] + [
         "", "| Test | Genes tested | Unadjusted p<0.05 | % | Pass BH 0.05 |", "|---|---|---|---|---|"] + [
         f"| {n} | {d['genes_tested']} | {d['n_unadjusted_lt_05']} | {d['pct_unadjusted_lt_05']} | {d['n_bh_lt_05']} |" for n, d in [
             ("Real labels", results["de"]["real"]), ("Shuffled, tie-corrected", s), ("Shuffled, no tie correction (secondary)", results["de"]["shuffled_no_tie_correction_secondary"])]]
(OUT / "results.md").write_text("\n".join(lines) + "\n")
results["predictions"]

[{'statement': 'more than 90% zero', 'held': True},
 {'statement': '6 to 10 clusters at 0.7', 'held': True},
 {'statement': 'at least 4 of 5 markers two-fold', 'held': True},
 {'statement': 'resolution 2.0 at least twice 0.25', 'held': True},
 {'statement': 'all seeds ARI at least 0.8', 'held': True},
 {'statement': 'shuffled 3-7% and at most 5 pass BH', 'held': False},
 {'statement': 'real labels at least 100 pass BH', 'held': True}]

## 10. Post-hoc: why did shuffled labels produce "significant" genes?

**Not on the run card.** Written after prediction 6 missed. Shuffled labels carry no signal, so any gene passing correction is a false alarm. This section looks at
which genes those were, and repeats the shuffle to see whether it was one unlucky draw.

In [11]:
_, shuffled_df = test(shuffled_groups)
counts = adata.layers["counts"]
total = pd.Series(np.asarray((counts > 0).sum(axis=0)).ravel(), index=adata.var_names)
in_a = pd.Series(np.asarray((counts[shuffled_groups] > 0).sum(axis=0)).ravel(), index=adata.var_names)
share_a = float(shuffled_groups.mean())
false_alarms = shuffled_df[shuffled_df.pvals_adj < 0.05].copy()
false_alarms = [{"gene": str(n), "cells_detected": int(total[n]), "in_group_a": int(in_a[n]), "reported_p": float(p)}
                for n, p in zip(false_alarms.names, false_alarms.pvals)]

RARE = 10
rare_genes = set(total[total < RARE].index)
repeats = []
for seed in range(1000, 1020):
    groups = np.random.default_rng(seed).permutation(real_groups)
    summary, df = test(groups)
    passed = df[df.pvals_adj < 0.05].names.astype(str)
    common = df[~df.names.astype(str).isin(rare_genes)]
    repeats.append({"seed": seed, "pct_lt_05": summary["pct_unadjusted_lt_05"], "n_bh": summary["n_bh_lt_05"],
                    "n_bh_rare": int(passed.isin(rare_genes).sum()), "n_bh_common": int((~passed.isin(rare_genes)).sum()),
                    "pct_lt_05_common": round(100 * float((common.pvals < 0.05).mean()), 2)})
rep = pd.DataFrame(repeats)
results["post_hoc"] = {"note": "not on the run card; written after prediction 6 missed",
                       "share_of_cells_in_group": round(share_a, 4), "rare_threshold_cells": RARE,
                       "genes_detected_in_under_threshold": len(rare_genes), "false_alarms_primary_shuffle": false_alarms,
                       "twenty_more_shuffles": {"pct_lt_05_min": float(rep.pct_lt_05.min()), "pct_lt_05_max": float(rep.pct_lt_05.max()),
                                                "n_bh_min": int(rep.n_bh.min()), "n_bh_median": float(rep.n_bh.median()), "n_bh_max": int(rep.n_bh.max()),
                                                "shuffles_with_any_bh": int((rep.n_bh > 0).sum()), "total_bh_rare": int(rep.n_bh_rare.sum()), "total_bh_common": int(rep.n_bh_common.sum()),
                                                "pct_lt_05_common_min": float(rep.pct_lt_05_common.min()), "pct_lt_05_common_max": float(rep.pct_lt_05_common.max())}}
(OUT / "results.json").write_text(json.dumps(results, indent=2) + "\n")
with open(OUT / "results.md", "a") as f:
    t = results["post_hoc"]["twenty_more_shuffles"]
    f.write("\n## Post-hoc, not on the run card\n\n")
    f.write("Genes passing correction on the primary shuffled labels:\n\n| Gene | Cells detected in | Of those, in the shuffled group | Reported p |\n|---|---|---|---|\n")
    f.write("".join(f"| {x['gene']} | {x['cells_detected']} | {x['in_group_a']} | {x['reported_p']:.2e} |\n" for x in false_alarms))
    f.write(f"\nTwenty more shuffles: {t['shuffles_with_any_bh']} of 20 had at least one gene pass correction (median {t['n_bh_median']}, max {t['n_bh_max']}). ")
    f.write(f"Of all genes passing, {t['total_bh_rare']} were detected in under {RARE} cells and {t['total_bh_common']} in {RARE} or more.\n")
results["post_hoc"]

{'note': 'not on the run card; written after prediction 6 missed',
 'share_of_cells_in_group': 0.1293,
 'rare_threshold_cells': 10,
 'genes_detected_in_under_threshold': 2619,
 'false_alarms_primary_shuffle': [{'gene': 'SLC7A6OS',
   'cells_detected': 82,
   'in_group_a': 25,
   'reported_p': 1.4095993775297543e-06},
  {'gene': 'PTRH2',
   'cells_detected': 49,
   'in_group_a': 17,
   'reported_p': 4.779156385718282e-06},
  {'gene': 'RP5-1142A6.2',
   'cells_detected': 3,
   'in_group_a': 3,
   'reported_p': 6.88998002645504e-06},
  {'gene': 'GTF2IRD1',
   'cells_detected': 3,
   'in_group_a': 3,
   'reported_p': 6.88998002645504e-06},
  {'gene': 'RAPH1',
   'cells_detected': 3,
   'in_group_a': 3,
   'reported_p': 6.88998002645504e-06},
  {'gene': 'RHOD',
   'cells_detected': 5,
   'in_group_a': 4,
   'reported_p': 7.718318125560496e-06},
  {'gene': 'CCDC9',
   'cells_detected': 27,
   'in_group_a': 11,
   'reported_p': 1.4380311372037884e-05}],
 'twenty_more_shuffles': {'pct_lt_05_mi

In [12]:
# Post-hoc, continued. A rough exact comparison: given how many cells detected the gene, how likely is at least this many landing in the shuffled group by chance?
# This uses detection only, not expression level, so it is a sanity check on the reported p and not a replacement test.
from scipy.stats import hypergeom
N, K = adata.n_obs, int(shuffled_groups.sum())
for x in results["post_hoc"]["false_alarms_primary_shuffle"]:
    x["exact_detection_p"] = float(hypergeom.sf(x["in_group_a"] - 1, N, K, x["cells_detected"]))
    x["bonferroni_exact"] = min(1.0, x["exact_detection_p"] * adata.n_vars)
results["post_hoc"]["genes_with_no_detection_after_qc"] = int((total == 0).sum())  # these give the divide warnings above; Scanpy reports them as not significant
(OUT / "results.json").write_text(json.dumps(results, indent=2) + "\n")
with open(OUT / "results.md", "a") as f:
    f.write("\n| Gene | Reported p | Exact detection p | Exact p x genes tested |\n|---|---|---|---|\n")
    f.write("".join(f"| {x['gene']} | {x['reported_p']:.2e} | {x['exact_detection_p']:.2e} | {x['bonferroni_exact']:.3g} |\n" for x in results["post_hoc"]["false_alarms_primary_shuffle"]))
results["post_hoc"]["false_alarms_primary_shuffle"]

[{'gene': 'SLC7A6OS',
  'cells_detected': 82,
  'in_group_a': 25,
  'reported_p': 1.4095993775297543e-06,
  'exact_detection_p': 1.7858708570769076e-05,
  'bonferroni_exact': 0.24491432933952711},
 {'gene': 'PTRH2',
  'cells_detected': 49,
  'in_group_a': 17,
  'reported_p': 4.779156385718282e-06,
  'exact_detection_p': 6.797880669724692e-05,
  'bonferroni_exact': 0.9322613550460442},
 {'gene': 'RP5-1142A6.2',
  'cells_detected': 3,
  'in_group_a': 3,
  'reported_p': 6.88998002645504e-06,
  'exact_detection_p': 0.002143397240817117,
  'bonferroni_exact': 1.0},
 {'gene': 'GTF2IRD1',
  'cells_detected': 3,
  'in_group_a': 3,
  'reported_p': 6.88998002645504e-06,
  'exact_detection_p': 0.002143397240817117,
  'bonferroni_exact': 1.0},
 {'gene': 'RAPH1',
  'cells_detected': 3,
  'in_group_a': 3,
  'reported_p': 6.88998002645504e-06,
  'exact_detection_p': 0.002143397240817117,
  'bonferroni_exact': 1.0},
 {'gene': 'RHOD',
  'cells_detected': 5,
  'in_group_a': 4,
  'reported_p': 7.71831812